In [48]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from qdrant_client.models import PointStruct
import tqdm as notebook_tqdm
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader, Docx2txtLoader


In [20]:
embedding_function = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-mpnet-base-v2"
)

In [28]:
# loading resumes
def doc_loader():
    loader = DirectoryLoader(
        path="../cv",
        glob="**/*.pdf",
        loader_cls=PyMuPDFLoader,
        show_progress = True
    )
    docs = loader.load()
    loader = DirectoryLoader(
        path="../cv",
        glob="**/*.docx",
        loader_cls=Docx2txtLoader,
        show_progress=True
    )
    docs.extend(loader.load())
    return docs

In [29]:
docs = doc_loader()
print(f"{len(docs)} number of documents loaded successfully")

100%|██████████| 5/5 [00:00<00:00, 159.51it/s]

12 number of documents loaded successfully


In [32]:
docs[0]

Document(metadata={'producer': 'pdfTeX-1.40.23', 'creator': 'TeX', 'creationdate': '2022-01-04T16:53:23+00:00', 'source': '../cv/AnuvaGoyal_Latex.pdf', 'file_path': '../cv/AnuvaGoyal_Latex.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2022-01-04T16:53:23+00:00', 'trapped': '', 'modDate': 'D:20220104165323Z', 'creationDate': 'D:20220104165323Z', 'page': 0}, page_content='ANUVA GOYAL\n[ anuvagoyal111@gmail.com\n½ Agra, Uttar Pradesh, India\n\x87 github.com/AnuvaGoyal\nPROJECTS\n• Mental Healthcare Chatbot that provides ad-\nvice to the user based on diﬀerent categories\nof mental health problems using a dataset\nwebscraped from counselchat.com (Nov 2021)\n• Full stack Speech Emotion based Movie\nRecommender System using the RAVDESS\nDataset and Web Scraping techniques (Oct\n2021)\n• Finding a Perfect Fit, a model to parse re-\nsumes using Pytesseract, NLP and XG Boost\nand Random Forest classiﬁcation techniques\n(Aug 20

In [41]:
# creating vector of cv_text
qdrant_client = QdrantClient(
    url='localhost:6333'
)
qdrant_client.create_collection(
    collection_name="cv_embeddings",
    vectors_config = VectorParams(size=768, distance=Distance.COSINE)
)
qdrant_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name="cv_embeddings",
    embedding=embedding_function
)

In [42]:
def ensure_cv_collection():
    try:
        qdrant_store = QdrantVectorStore(
            client=qdrant_client,
            collection_name='cv_embeddings',
            embedding=embedding_function
        )
        print(f"Collection is already is present")
    except:
        qdrant_client.create_collection(
            collection_name='cv_embeddings',
            vectors_config=VectorParams(size=768, distance=Distance.COSINE)
        )
        print(f"New Collection Created")

In [43]:
ensure_cv_collection()

Collection is already is present


In [61]:
import hashlib
import uuid
def get_file_hash(file_path):
    with open(file_path, 'rb') as f:
        hex_digest = hashlib.sha256(f.read()).hexdigest()
        return str(uuid.UUID(hex_digest[:32]))

In [62]:
get_file_hash("/home/anujkumar/resumeRanking/cv/1901841_RESUME.pdf")

'329ff176-515b-32bc-d21c-aaf37ec85c62'

In [63]:
"/home/anujkumar/resumeRanking/cv/1901841_RESUME.pdf".endswith(".pdf")

True

In [64]:
def upload_cv(cv_path):
    if not os.path.exists(cv_path):
        return "Path doesn't exists."
    if cv_path.endswith('.pdf'):
        loader=PyMuPDFLoader(cv_path)
    elif cv_path.endswith('.docx') or cv_path.endswith('.doc'):
        loader=Docx2txtLoader(cv_path)
    else:
        return "File Format is not supported. Please provide pdf or docx"
    
    docs = loader.load()
    cv_text = docs[0].page_content
    cv_vec = embedding_function.embed_query(cv_text)

    payload = {
        "total_exp": 0,
        "exp_txt": "Exp section"
    }
    point = PointStruct(
        id=get_file_hash(cv_path),
        vector = cv_vec,
        payload = payload
    )

    qdrant_client.upsert(
        collection_name='cv_embeddings',
        points = [point]
    )
    return f"{cv_path} has been loaded with in the cv_embeddings collection"
    

In [65]:
upload_cv("/home/anujkumar/resumeRanking/cv/react-developer-resume-example.pdf")

'/home/anujkumar/resumeRanking/cv/react-developer-resume-example.pdf has been loaded with in the cv_embeddings collection'